# Document Intelligence: Extract Data from Documents

In this notebook you will learn how to use **Azure AI Document Intelligence** to:

1. Read and extract text from a PDF using the **prebuilt-read** model
2. Extract structured fields from an **invoice**
3. Extract structured fields from a **receipt**
4. Detect **handwritten text** and measure confidence

---

### Prerequisites: Deploy the Azure Resource

Before running this notebook, you need an **Azure AI Document Intelligence** resource. Instead of configuring it manually in the Azure Portal, deploy the included **ARM template** which creates the resource in the correct configuration automatically.

> **Tip:** Open `deploy-document-intelligence.parameters.json` to customize the resource name, region, or SKU before deploying. The default SKU is **S0** (paid). Change it to **F0** for the free tier.

**1. Deploy the resource** — run the cell below (replace `<your-resource-group>`):

In [ ]:
import subprocess, shutil

# ⚠️ Replace <your-resource-group> with your actual resource group name
RESOURCE_GROUP = "Foundry"

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", "deploy-document-intelligence.json",
     "--parameters", "deploy-document-intelligence.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

**2. Assign the required RBAC role** so your identity can use the resource (replace `<your-resource-group>`):

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a 401 error on your first run, wait a moment and try again.

In [ ]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the resource ID from the deployment output
scope_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-document-intelligence",
     "--query", "properties.outputs.resourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
resource_id = scope_result.stdout.strip()
print(f"Resource ID:  {resource_id}")

# Step C: Assign the RBAC role
role_result = subprocess.run(
    [AZ, "role", "assignment", "create",
     "--assignee", user_id,
     "--role", "Cognitive Services User",
     "--scope", resource_id],
    capture_output=True, text=True, encoding="utf-8"
)
print(role_result.stdout)
if role_result.returncode != 0:
    print("ERROR:", role_result.stderr)

**3. Update the `.env` file** in this folder with the endpoint from the deployment output:

```
DOCUMENT_INTELLIGENCE_ENDPOINT="https://<your-account-name>.cognitiveservices.azure.com/"
```

You can retrieve this value by running the cell below (replace `<your-resource-group>`):

In [ ]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-document-intelligence",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

> **Note:** The demos below use publicly hosted sample documents — no local files needed.

## Step 1: Install the Document Intelligence SDK

In [ ]:
%pip install azure-ai-documentintelligence azure-identity python-dotenv

## Step 2: Load Environment Variables and Create the Client

In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult

load_dotenv(override=True)

doc_endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")

# Authenticate via Entra ID (uses az login credentials)
credential = DefaultAzureCredential()

doc_client = DocumentIntelligenceClient(
    endpoint=doc_endpoint,
    credential=credential,
)

print("Endpoint:", doc_endpoint)
print("Client ready:", doc_client is not None)

## Step 3: Read Model -- Extract Text from a PDF

The `prebuilt-read` model is a general-purpose text extractor. It works on
PDFs, images, and scanned documents, returning paragraphs, lines, and words.

In [ ]:
# A publicly available sample image from Azure documentation
pdf_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/rest-api/read.png"

# Send the document for analysis
poller = doc_client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=pdf_url),
)
read_result: AnalyzeResult = poller.result()

# Print each paragraph the service found
print(f"Found {len(read_result.paragraphs)} paragraph(s):\n")
for i, para in enumerate(read_result.paragraphs, start=1):
    print(f"Paragraph {i}: {para.content}\n")

## Step 4: Invoice Model -- Extract Invoice Fields

The `prebuilt-invoice` model understands invoice layouts and can
automatically extract fields like customer name, invoice ID, totals, etc.

In [ ]:
# Sample invoice from Azure documentation
invoice_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"

invoice_poller = doc_client.begin_analyze_document(
    "prebuilt-invoice",
    AnalyzeDocumentRequest(url_source=invoice_url),
)
invoice_result = invoice_poller.result()

for invoice in invoice_result.documents:
    fields = invoice.fields

    # Safely extract common invoice fields
    vendor = fields.get("VendorName", {})
    inv_id = fields.get("InvoiceId", {})
    total  = fields.get("InvoiceTotal", {})
    date   = fields.get("InvoiceDate", {})

    print(f"Vendor:       {vendor.get('content', 'N/A')}")
    print(f"Invoice ID:   {inv_id.get('content', 'N/A')}")
    print(f"Total:        {total.get('content', 'N/A')}")
    print(f"Date:         {date.get('content', 'N/A')}")

## Step 5: Receipt Model -- Extract Receipt Fields

The `prebuilt-receipt` model is optimized for retail receipts and can
pull out the merchant name, total, transaction date, and line items.

In [ ]:
# Sample receipt from Azure documentation
receipt_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/contoso-receipt.png"

receipt_poller = doc_client.begin_analyze_document(
    "prebuilt-receipt",
    AnalyzeDocumentRequest(url_source=receipt_url),
)
receipt_result = receipt_poller.result()

for receipt in receipt_result.documents:
    fields = receipt.fields

    merchant = fields.get("MerchantName", {})
    total    = fields.get("Total", {})
    txn_date = fields.get("TransactionDate", {})

    print(f"Merchant:         {merchant.get('content', 'N/A')}")
    print(f"Total:            {total.get('content', 'N/A')}")
    print(f"Transaction Date: {txn_date.get('content', 'N/A')}")

## Step 6: Detect Handwritten Text

The `prebuilt-read` model can also detect whether text is **handwritten**
and reports a confidence score. This is useful for digitizing forms,
notes, or letters.

In [ ]:
# A sample form containing handwritten text (from the Azure SDK test suite)
handwritten_url = "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/forms/Form_1.jpg"

hw_poller = doc_client.begin_analyze_document(
    "prebuilt-read",
    AnalyzeDocumentRequest(url_source=handwritten_url),
)
hw_result: AnalyzeResult = hw_poller.result()

# Check style information for handwriting detection
if hw_result.styles:
    for style in hw_result.styles:
        if style.is_handwritten:
            print(f"Handwritten text detected (confidence: {style.confidence:.0%})")
else:
    print("No style information returned.")

# Print the extracted paragraphs regardless
print("\nExtracted text:")
for i, para in enumerate(hw_result.paragraphs, start=1):
    print(f"  {i}. {para.content}")